In [4]:
import asyncio
from openai import AsyncOpenAI
from agents import Agent, Runner, OpenAIChatCompletionsModel
from agents.tracing import set_tracing_disabled

set_tracing_disabled(True)

ollama_client = AsyncOpenAI(
    base_url="http://localhost:11434/v1",   # Ollama's OpenAI-compatible endpoint
    api_key="ollama"                        # Dummy key — Ollama doesn't need one
)

local_model = OpenAIChatCompletionsModel(
    model="qwen3.5:cloud ",
    openai_client=AsyncOpenAI(
        api_key="ollama",
        base_url="http://localhost:11434/v1",
    )
)

agent = Agent(
    name="Local Assistant",
    instructions="You are a helpful assistant running locally via Ollama. Be concise.",
    model=local_model             # <-- This is the key line!
)

result = await Runner.run(agent, "What is the Agents SDK in one paragraph?")
print(result.final_output)

In [ ]:
from openai import AsyncOpenAI
from agents import (
    Agent, Model, ModelProvider,
    OpenAIChatCompletionsModel, Runner,
    set_tracing_disabled
)

set_tracing_disabled(True)

OLLAMA_BASE_URL = "http://localhost:11434/v1"
OLLAMA_MODEL = "qwen3.5:cloud"


class OllamaProvider(ModelProvider):
    """Custom ModelProvider that routes all model requests to Ollama."""
    
    def __init__(self, model_name: str = OLLAMA_MODEL):
        self.model_name = model_name
        self.client = AsyncOpenAI(
            base_url=OLLAMA_BASE_URL,
            api_key="ollama"
        )
    
    def get_model(self, model_name: str | None = None) -> Model:
        return OpenAIChatCompletionsModel(
            model=model_name or self.model_name,
            openai_client=self.client
        )


# Now create agents WITHOUT specifying model each time
ollama = OllamaProvider()

agent = Agent(
    name="Local Agent",
    instructions="You are a local AI assistant. Be helpful and concise.",
    model=ollama.get_model()  # Clean and reusable!
)

result = await Runner.run(agent, "Explain what Ollama is in 2 sentences.")
print(result.final_output)

NotFoundError: Error code: 404 - {'error': {'message': "model 'llama3.1:8b' not found", 'type': 'not_found_error', 'param': None, 'code': None}}